# Collect US PCE data

References : [Chapter 5 : PCE (BEA)](https://www.bea.gov/resources/methodologies/nipa-handbook/pdf/chapter-05.pdf)

Careful to NIPA levels ; the same tables 2.4.3, 2.4.4 and 2.4.5 without the "U" that stands for "Underlying Details" are different aggregation levels. Those one are the most frequently used, while Lansing and Shapiro use the "Underlying Details" tables (129 components).

In [1]:
# Librairies
import json
import pandas as pd
from fismi import sdxData as sdx
import numpy as np

In [2]:
with open('/Users/lea_gosselin/.openbb_platform/user_settings.json', 'r') as file:
    keys = json.load(file)

In [16]:
url = "https://apps.bea.gov/api/data"
userid = keys["credentials"].get("bea_api_key")

# Load PCE data from BEA

# Query data
index_raw = sdx.getBeaData("U20403", userid)   # Table 2.4.3U : Real Personal Consumption Expenditures by Type of Product, Quantity Indexes
price_raw   = sdx.getBeaData("U20404", userid)   # Table 2.4.4U : Price Indexes for Personal Consumption Expenditures by Type of Product
weights_raw = sdx.getBeaData("U20405", userid)   # Table 2.4.5U : Personal Consumption Expenditures by Type of Product


In [4]:
# Weights PCE (share of the expenditure in the total, as current $) 
df_price   = price_raw.drop_duplicates(subset=["TIME_PERIOD","SeriesCode"]).pivot(index="TIME_PERIOD", columns="LineDescription", values="DataValue")
df_weights = weights_raw.drop_duplicates(subset=["TIME_PERIOD","SeriesCode"]).pivot(index="TIME_PERIOD", columns="SeriesCode", values="DataValue")

In [51]:
import re
seriesCode = pd.DataFrame(index_raw["LineDescription"].unique(), columns=["Code"])
digits = [''.join(re.findall(r'\d', s)) for s in seriesCode["Code"]]
seriesCode["Digits"] = digits
seriesCode

,Code,Digits
0,Personal consumption expenditures,
1,Goods,
2,Durable goods,
3,Motor vehicles and parts,
4,New motor vehicles (55),55
...,...,...
393,Market-based PCE household maintenance,
394,Market-based PCE food and energy,
395,Market-based PCE excluding food,
396,Market-based PCE excluding energy,


In [52]:
seriesCode[seriesCode["Digits"] != ""]

,Code,Digits
4,New motor vehicles (55),55
11,Net purchases of used motor vehicles (56),56
19,Motor vehicle parts and accessories (58),58
23,Furniture and furnishings (parts of 31 and 32),3132
28,Household appliances (part of 33),33
...,...,...
331,Foreign travel by U.S. residents (129),129
335,Less: Expenditures in the United States by non...,130
339,Final consumption expenditures of nonprofit in...,132
340,Gross output of nonprofit institutions (133),133


In [53]:
BEA_URL = "https://apps.bea.gov/api/data/"

import requests

params = {
    "UserID": userid,
    "method": "GetData",
    "datasetname": "NIUnderlyingDetail",
    "TableName": "U20403",
    "Frequency": "M",
    "Year": "ALL",
    "ResultFormat": "JSON",
}
r = requests.get(BEA_URL, params=params, timeout=60)
r.raise_for_status()
payload = r.json() # ["BEAAPI"]["Results"]